# 04 - t-SNE and UMAP

**Author:** Beyza Şkr  
**Based on:** *AI in Chemical Engineering: Unlocking the Power Within Data* (Romagnoli et al.)

PCA is linear. It finds directions of maximum variance, but it can miss curved / local structure.

t-SNE and UMAP are **nonlinear** dimensionality reduction methods.  
They are mostly used for **visualization**, not as a preprocessing step for every model.

In this notebook:
- create and scale synthetic process data
- project with PCA (as a baseline)
- project with t-SNE
- project with UMAP
- compare the three plots


## 0. Libraries

UMAP is not inside scikit-learn. In Colab run the install cell once.


In [ ]:
# Run this cell once in Colab if umap is not installed
# !pip install umap-learn -q


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

import umap

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (8, 6)
print("Libraries ready.")


## 1. Synthetic process data

Same idea as before: temperature, pressure, flow rate, concentration.

I also add a simple operating regime label so the plots are easier to interpret.  
This label is **not** given to PCA / t-SNE / UMAP. It is only used for coloring.


In [ ]:
np.random.seed(42)
n = 400

# two rough operating regimes
regime = np.random.choice([0, 1], size=n, p=[0.55, 0.45])

temperature = np.where(
    regime == 0,
    np.random.normal(82, 6, n),
    np.random.normal(95, 6, n),
)
pressure = np.where(
    regime == 0,
    np.random.normal(2.3, 0.25, n),
    np.random.normal(2.8, 0.25, n),
)
flow_rate = np.where(
    regime == 0,
    np.random.normal(110, 10, n),
    np.random.normal(135, 10, n),
)
concentration = np.where(
    regime == 0,
    np.random.normal(0.40, 0.05, n),
    np.random.normal(0.52, 0.05, n),
)

df = pd.DataFrame({
    "temperature": temperature,
    "pressure": pressure,
    "flow_rate": flow_rate,
    "concentration": concentration,
    "regime": regime,
})

numeric_cols = ["temperature", "pressure", "flow_rate", "concentration"]

print("Shape:", df.shape)
print(df.head())
print("\nRegime counts:")
print(df["regime"].value_counts())


## 2. Scale the data

All three methods care about distance.  
If we do not scale, flow_rate (~110-135) will dominate concentration (~0.4-0.5).


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numeric_cols])

print("Mean (should be ~0):", np.round(X_scaled.mean(axis=0), 4))
print("Std  (should be ~1):", np.round(X_scaled.std(axis=0), 4))


## 3. PCA baseline

PCA is linear and fast.  
We keep it as a reference so we can see what t-SNE / UMAP add.


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_.round(3))
print("Total variance kept by 2 PCs:", f"{pca.explained_variance_ratio_.sum():.1%}")


In [ ]:
plt.figure()
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["regime"], cmap="viridis", alpha=0.75, edgecolor="k", linewidth=0.2)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA")
plt.colorbar(label="Regime (only for color)")
plt.grid(True, alpha=0.3)
plt.show()


## 4. t-SNE

t-SNE tries to keep **local neighborhoods**.  
Points that are close in high dimension should stay close in 2D.

Important parameters:
- `n_components=2` → we want a 2D plot
- `perplexity` → roughly "how many neighbors should I care about?"  
  Typical values: 5 to 50. Common starting point: 30.
- `random_state` → so the plot is reproducible
- t-SNE has **no transform()** for new points. You fit it on the data you want to plot.

t-SNE is good for visualization.  
It is not a good default input for clustering / classification pipelines.


In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    random_state=42,
    init="pca",
)

X_tsne = tsne.fit_transform(X_scaled)

print("t-SNE result shape:", X_tsne.shape)
print("First 5 rows:")
print(X_tsne[:5])


`X_tsne[:, 0]` → x values (first column)  
`X_tsne[:, 1]` → y values (second column)


In [ ]:
plt.figure()
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=df["regime"], cmap="viridis", alpha=0.75, edgecolor="k", linewidth=0.2)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE (perplexity=30)")
plt.colorbar(label="Regime (only for color)")
plt.grid(True, alpha=0.3)
plt.show()


### Effect of perplexity

If the plot looks like one blob, perplexity may be too high.  
If it looks like many tiny islands, perplexity may be too low.

This cell just compares two values. No need to overthink it.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, perp in zip(axes, [10, 50]):
    X_tmp = TSNE(n_components=2, perplexity=perp, random_state=42, init="pca").fit_transform(X_scaled)
    ax.scatter(X_tmp[:, 0], X_tmp[:, 1], c=df["regime"], cmap="viridis", alpha=0.75, s=18)
    ax.set_title(f"t-SNE, perplexity={perp}")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. UMAP

UMAP is also nonlinear, but usually:
- faster than t-SNE on larger data
- better at keeping some global structure
- has parameters that are a bit more intuitive

Important parameters:
- `n_neighbors` → local vs global balance. Small = more local. Large = more global.
- `min_dist` → how tightly points are packed in the 2D plot
- `n_components=2`
- `random_state` for reproducibility

Like t-SNE, UMAP is mainly a **visualization** tool here.


In [ ]:
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42,
)

X_umap = reducer.fit_transform(X_scaled)

print("UMAP result shape:", X_umap.shape)
print("First 5 rows:")
print(X_umap[:5])


In [ ]:
plt.figure()
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=df["regime"], cmap="viridis", alpha=0.75, edgecolor="k", linewidth=0.2)
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP (n_neighbors=15, min_dist=0.1)")
plt.colorbar(label="Regime (only for color)")
plt.grid(True, alpha=0.3)
plt.show()


## 6. Compare PCA, t-SNE and UMAP

Same data, same colors, three different projections.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

plots = [
    (X_pca, "PCA"),
    (X_tsne, "t-SNE"),
    (X_umap, "UMAP"),
]

for ax, (X_2d, title) in zip(axes, plots):
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=df["regime"], cmap="viridis", alpha=0.75, s=18)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### What to notice

- PCA: linear, axes have meaning (variance). Good first look.
- t-SNE: local clusters can look very clear, but distances between far groups are not reliable.
- UMAP: often a middle ground. Local groups + a bit more global layout.

Coloring by `regime` is only to check whether the 2D map recovered a structure that we already know exists.  
In a real unlabeled process dataset you would not have this color.


## 7. Optional: color by KMeans instead of the true regime

In unsupervised work we usually do not have a true label.  
One common check is: cluster the scaled data, then color the t-SNE / UMAP plot with those cluster IDs.


In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

print("KMeans cluster counts:")
print(pd.Series(clusters).value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=clusters, cmap="viridis", alpha=0.75, s=18)
axes[0].set_title("t-SNE colored by KMeans")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_umap[:, 0], X_umap[:, 1], c=clusters, cmap="viridis", alpha=0.75, s=18)
axes[1].set_title("UMAP colored by KMeans")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Summary

- Scale first. Always.
- PCA = linear baseline, interpretable variance.
- t-SNE = local structure, mainly for plots. Tune `perplexity`.
- UMAP = also for plots, usually faster, tune `n_neighbors` and `min_dist`.
- Do not treat t-SNE / UMAP coordinates as true physical features of the process.

**Next notebook:** KMeans + Elbow + Silhouette
